# Classificação de dados estruturados do zero

**Base:** exemplo oficial do Keras ["Structured data classification from scratch"](https://keras.io/examples/structured_data/structured_data_classification_from_scratch/) (fchollet).<br>
**Descrição:** classificação binária de dados tabulares com features numéricas e categóricas.

## Introdução

Neste notebook fazemos a classificação de dados estruturados a partir de um arquivo CSV bruto.
Os dados têm features **numéricas** e **categóricas**. Usamos camadas de pré-processamento do Keras
para normalizar as features numéricas e transformar as categóricas em vetores (one-hot).

### O dataset

O [dataset](https://archive.ics.uci.edu/ml/datasets/heart+Disease) é da Cleveland Clinic Foundation
for Heart Disease. É um CSV com 303 linhas: cada linha é um paciente (**amostra**) e cada coluna é um
atributo do paciente (**feature**). O objetivo é prever se o paciente tem doença cardíaca
(**classificação binária**).

Coluna | Descrição | Tipo
------------|--------------------|----------------------
Age | Idade em anos | Numérica
Sex | (1 = masculino; 0 = feminino) | Categórica
CP | Tipo de dor no peito (0, 1, 2, 3, 4) | Categórica
Trestbpd | Pressão arterial em repouso (mm Hg na admissão) | Numérica
Chol | Colesterol sérico em mg/dl | Numérica
FBS | Glicemia em jejum > 120 mg/dl (1 = sim; 0 = não) | Categórica
RestECG | Resultado do eletrocardiograma em repouso (0, 1, 2) | Categórica
Thalach | Frequência cardíaca máxima atingida | Numérica
Exang | Angina induzida por exercício (1 = sim; 0 = não) | Categórica
Oldpeak | Depressão do segmento ST induzida por exercício | Numérica
Slope | Inclinação do segmento ST no pico do exercício | Numérica
CA | Número de vasos principais (0-3) coloridos por fluoroscopia | Numérica e categórica
Thal | 3 = normal; 6 = defeito fixo; 7 = defeito reversível | Categórica
Target | Diagnóstico de doença cardíaca (1 = sim; 0 = não) | Alvo

## Configuração

In [1]:
import os

# Define o TensorFlow como backend do Keras (precisa vir antes de importar o keras)
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers
import pandas as pd
import tensorflow as tf  # usado para montar o tf.data.Dataset

## Preparando os dados

Baixamos o CSV e carregamos em um DataFrame do Pandas:

In [2]:
file_url = "http://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
dataframe = pd.read_csv(file_url)

O dataset tem 303 amostras e 14 colunas (13 features + o alvo):

In [3]:
dataframe.shape

(303, 14)

Prévia de algumas amostras:

In [4]:
dataframe.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0,normal,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,normal,0


A última coluna, `target`, indica se o paciente tem doença cardíaca (1) ou não (0).

Separamos os dados em treino e validação (80% / 20%):

In [5]:
# 20% das linhas vão para validação; random_state fixo deixa a divisão reproduzível
val_dataframe = dataframe.sample(frac=0.2, random_state=1337)

# O restante (linhas que não foram sorteadas) fica para treino
train_dataframe = dataframe.drop(val_dataframe.index)

print(
    f"Usando {len(train_dataframe)} amostras para treino "
    f"e {len(val_dataframe)} para validação"
)

Usando 242 amostras para treino e 61 para validação


## Metadados do dataset

Aqui definimos quais colunas são numéricas, quais são categóricas e qual é o alvo.
Essas listas são usadas para ler os dados e codificar cada feature de acordo com o seu tipo.

In [6]:
COLUMN_NAMES = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target",
]

# Nome da coluna alvo
TARGET_FEATURE_NAME = "target"

# Features numéricas (serão normalizadas)
NUMERIC_FEATURE_NAMES = ["age", "trestbps", "thalach", "oldpeak", "slope", "chol"]

# Features categóricas e o vocabulário (valores possíveis) de cada uma.
# Todas as colunas que não são numéricas nem o alvo entram aqui.
CATEGORICAL_FEATURES_WITH_VOCABULARY = {
    feature_name: sorted(
        [
            # Categorias inteiras continuam int; categorias de texto viram str
            value if dataframe[feature_name].dtype == "int64" else str(value)
            for value in list(dataframe[feature_name].unique())
        ]
    )
    for feature_name in COLUMN_NAMES
    if feature_name not in list(NUMERIC_FEATURE_NAMES + [TARGET_FEATURE_NAME])
}

# Lista com o nome de todas as features usadas pelo modelo
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + list(CATEGORICAL_FEATURES_WITH_VOCABULARY.keys())

print("Numéricas:", NUMERIC_FEATURE_NAMES)
print("Categóricas:", list(CATEGORICAL_FEATURES_WITH_VOCABULARY.keys()))

Numéricas: ['age', 'trestbps', 'thalach', 'oldpeak', 'slope', 'chol']
Categóricas: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal']


## Pré-processamento com camadas do Keras

As features categóricas codificadas como inteiros são: `sex`, `cp`, `fbs`, `restecg`, `exang` e `ca`.
Elas são transformadas com **one-hot encoding** usando `IntegerLookup()`.

A feature `thal` é categórica, mas em texto. Para ela usamos `StringLookup()`.

As features numéricas contínuas (`age`, `trestbps`, `chol`, `thalach`, `oldpeak` e `slope`)
passam por uma camada `Normalization()`, que deixa cada uma com média 0 e desvio padrão 1.

Definimos duas funções auxiliares:

- `encode_categorical`: aplica o one-hot nas features categóricas de cada amostra;
- `encode_numerical_feature`: aprende a média/desvio do treino e normaliza a feature numérica.

In [7]:
def encode_categorical(features, target):
    """Converte as features categóricas de uma amostra em vetores one-hot."""
    for feature_name in features:
        if feature_name in CATEGORICAL_FEATURES_WITH_VOCABULARY:
            # Texto usa StringLookup; número inteiro usa IntegerLookup
            lookup_class = (
                layers.StringLookup
                if features[feature_name].dtype == "string"
                else layers.IntegerLookup
            )
            vocabulary = CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name]

            # Camada que mapeia cada categoria para uma posição do vetor one-hot.
            # Sem token de máscara e sem índice para valores desconhecidos (oov).
            index = lookup_class(
                vocabulary=vocabulary,
                mask_token=None,
                num_oov_indices=0,
                output_mode="binary",
            )
            features[feature_name] = index(features[feature_name])

    # Retorna um dict comum para combinar com as entradas do modelo
    return dict(features), target


def encode_numerical_feature(feature, name, dataset):
    """Normaliza uma feature numérica usando as estatísticas do dataset de treino."""
    normalizer = layers.Normalization()

    # Dataset só com a feature desejada, no formato (amostras, 1)
    feature_ds = dataset.map(lambda x, y: x[name])
    feature_ds = feature_ds.map(lambda x: tf.expand_dims(x, -1))

    # Aprende média e desvio padrão da feature
    normalizer.adapt(feature_ds)

    # Aplica a normalização na entrada simbólica do modelo
    return normalizer(feature)

Criamos um `tf.data.Dataset` para cada DataFrame:

In [8]:
def dataframe_to_dataset(dataframe):
    """Transforma um DataFrame em tf.data.Dataset de (features, alvo)."""
    dataframe = dataframe.copy()
    labels = dataframe.pop("target")  # separa o alvo das features
    ds = tf.data.Dataset.from_tensor_slices((dict(dataframe), labels)).map(encode_categorical)
    ds = ds.shuffle(buffer_size=len(dataframe))  # embaralha as amostras
    return ds


train_ds = dataframe_to_dataset(train_dataframe)
val_ds = dataframe_to_dataset(val_dataframe)

Cada item do `Dataset` é uma tupla `(entrada, alvo)`, onde a entrada é um dicionário de features e o alvo é `0` ou `1`:

In [9]:
for x, y in train_ds.take(1):
    print("Entrada:", x)
    print("Alvo:", y)

Entrada: {'age': <tf.Tensor: shape=(), dtype=int64, numpy=66>, 'sex': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([1, 0])>, 'cp': <tf.Tensor: shape=(5,), dtype=int64, numpy=array([0, 0, 0, 0, 1])>, 'trestbps': <tf.Tensor: shape=(), dtype=int64, numpy=178>, 'chol': <tf.Tensor: shape=(), dtype=int64, numpy=228>, 'fbs': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([0, 1])>, 'restecg': <tf.Tensor: shape=(3,), dtype=int64, numpy=array([1, 0, 0])>, 'thalach': <tf.Tensor: shape=(), dtype=int64, numpy=165>, 'exang': <tf.Tensor: shape=(2,), dtype=int64, numpy=array([0, 1])>, 'oldpeak': <tf.Tensor: shape=(), dtype=float64, numpy=1.0>, 'slope': <tf.Tensor: shape=(), dtype=int64, numpy=2>, 'ca': <tf.Tensor: shape=(4,), dtype=int64, numpy=array([0, 0, 1, 0])>, 'thal': <tf.Tensor: shape=(5,), dtype=int64, numpy=array([0, 0, 0, 0, 1])>}
Alvo: tf.Tensor(1, shape=(), dtype=int64)


Agrupamos os dados em lotes (batches) de 32 amostras:

In [10]:
train_ds = train_ds.batch(32)
val_ds = val_ds.batch(32)

## Construindo o modelo

Com os dados prontos, criamos o modelo de ponta a ponta. A rede concatena todas as features,
passa por uma camada `Dense(32)` com ReLU, aplica `Dropout(0.5)` para reduzir overfitting e termina
em um neurônio com sigmoid (probabilidade de doença cardíaca).

In [11]:
def create_model_inputs():
    """Cria uma entrada do Keras para cada feature, com o formato certo para cada tipo."""
    inputs = {}

    for feature_name in FEATURE_NAMES:
        if feature_name in CATEGORICAL_FEATURES_WITH_VOCABULARY:
            # Categórica: vetor one-hot com tamanho igual ao número de categorias
            num_categories = len(CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name])
            inputs[feature_name] = layers.Input(
                name=feature_name, shape=(num_categories,), dtype="int64"
            )
        else:
            # Numérica: um valor real por amostra, já normalizado
            feature_input = layers.Input(name=feature_name, shape=(1,), dtype="float32")
            inputs[feature_name] = encode_numerical_feature(feature_input, feature_name, train_ds)
    return inputs


class Classifier(keras.layers.Layer):
    """Camada com a lógica de classificação: Dense(32) -> Dropout -> Dense(1)."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.dense_1 = layers.Dense(32, activation="relu")
        self.dropout = layers.Dropout(0.5)
        self.dense_2 = layers.Dense(1, activation="sigmoid")

    def call(self, inputs):
        # Junta todas as features em um único vetor
        all_features = layers.concatenate(list(inputs.values()))
        x = self.dense_1(all_features)
        x = self.dropout(x)
        return self.dense_2(x)

    def build(self, input_shape):
        # Evita avisos de build: as camadas internas são construídas no primeiro uso
        self.built = True


def create_model():
    """Monta o modelo completo a partir das entradas e do classificador."""
    all_inputs = create_model_inputs()
    output = Classifier()(all_inputs)
    return keras.Model(all_inputs, output)


model = create_model()

# Adam como otimizador, Binary Cross-Entropy como loss e acurácia como métrica
model.compile("adam", "binary_crossentropy", metrics=["accuracy"])

C:\3 - AI GROUP\machine_learning_2026.2\deep_learning\.venv\Lib\site-packages\keras\src\models\functional.py:106: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'age' mapping to value <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor> which has name 'keras_tensor'. Change the tensor name to 'age' (via `Input(..., name='age')`)
  warnings.warn(
C:\3 - AI GROUP\machine_learning_2026.2\deep_learning\.venv\Lib\site-packages\keras\src\models\functional.py:106: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'trestbps' mapping to value <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1> which has name 'keras_tensor_1'. Change the tensor name to 'trestbps' (via `Input(..., name='trestbps')`)
  warnings.warn(
C:\3 - AI GROUP\machine_learning_20

Visualizando o grafo de conexões do modelo:

In [12]:
# rankdir="LR" deixa o grafo na horizontal.
# O plot_model precisa do Graphviz instalado no sistema; sem ele, só mostramos um aviso
# e o notebook continua normalmente.
try:
    grafo = keras.utils.plot_model(model, show_shapes=True, rankdir="LR")
except ImportError as erro:
    grafo = None
    print("Grafo não gerado:", erro)

grafo

Grafo não gerado: You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


## Treinando o modelo

In [13]:
# 50 épocas, avaliando na validação ao final de cada uma
history = model.fit(train_ds, epochs=50, validation_data=val_ds)

Epoch 1/50


C:\3 - AI GROUP\machine_learning_2026.2\deep_learning\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1/8 ━━━━━━━━━━━━━━━━━━━━ 4s 678ms/step - accuracy: 0.5312 - loss: 14.6297

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.6033 - loss: 10.1740 - val_accuracy: 0.7705 - val_loss: 4.4867


Epoch 2/50


...

1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.7188 - loss: 0.6324

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7273 - loss: 0.5693 - val_accuracy: 0.8197 - val_loss: 0.4263


Epoch 50/50


1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.8438 - loss: 0.4382

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7397 - loss: 0.5160 - val_accuracy: 0.7705 - val_loss: 0.4329


## Inferência em novos dados

Para prever uma nova amostra usamos `model.predict()`. Antes, é preciso:

1. colocar cada valor dentro de uma lista, para ter a dimensão de batch (o modelo processa lotes, não amostras soltas);
2. aplicar o mesmo one-hot das categóricas e converter cada feature em tensor.

In [14]:
# Novo paciente que queremos avaliar
sample = {
    "age": 60,
    "sex": 1,
    "cp": 1,
    "trestbps": 145,
    "chol": 233,
    "fbs": 1,
    "restecg": 2,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 2.3,
    "slope": 3,
    "ca": 0,
    "thal": "fixed",
}


def get_cat_encoding(cat, cat_value):
    """Retorna o vetor one-hot de um valor dentro da lista de categorias."""
    encoding = [0] * len(cat)  # vetor de zeros com uma posição por categoria
    if cat_value in cat:
        encoding[cat.index(cat_value)] = 1  # marca 1 na posição da categoria
    return encoding


# Aplica o one-hot nas features categóricas da amostra
for name, value in sample.items():
    if name in CATEGORICAL_FEATURES_WITH_VOCABULARY:
        sample[name] = get_cat_encoding(CATEGORICAL_FEATURES_WITH_VOCABULARY[name], value)

# Converte cada feature em tensor com dimensão de batch
input_dict = {name: tf.convert_to_tensor([value]) for name, value in sample.items()}
predictions = model.predict(input_dict)

print(
    f"Este paciente tem {100 * predictions[0][0]:.1f}% de probabilidade "
    "de ter doença cardíaca, segundo o modelo."
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


Este paciente tem 32.1% de probabilidade de ter doença cardíaca, segundo o modelo.


## Conclusões

- O pré-processamento fica dentro do pipeline do Keras: `Normalization` para as numéricas e `IntegerLookup`/`StringLookup` para as categóricas.
- Com uma rede pequena (`Dense(32)` + `Dropout` + `Dense(1)`), o modelo converge rápido e a acurácia de validação fica em torno de 80%, como no exemplo original.
- Com só 61 amostras de validação, cada acerto ou erro muda a acurácia em cerca de 1,6 ponto percentual, então os valores oscilam entre as épocas.
- A saída é uma **probabilidade**; para obter a classe ainda é preciso aplicar um threshold (como na atividade anterior).